# LeetCode 3812 — Minimum Edge Toggles on a Tree bằng Python

- Link: https://leetcode.com/problems/minimum-edge-toggles-on-a-tree/
- Độ khó: **Hard**
- Chủ đề: Cây, tham lam từ lá lên (DFS hậu thứ tự), tính chẵn lẻ trong cây con, XOR, tính **duy nhất** của lời giải trên cây

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumFlips(self, n: int, edges: List[List[int]], start: str, target: str) -> List[int]:
```

## 1. Mục tiêu học

1. Nhận ra mỗi cạnh chỉ cần dùng **0 hoặc 1 lần** (dùng hai lần thì triệt tiêu) và thứ tự thao tác **không quan trọng** (phép lật là XOR).
2. Chứng minh trên cây, tập cạnh cần lật là **duy nhất**: cạnh `(con, cha)` được lật ⇔ số nút "lệch" trong cây con của `con` là **lẻ** ⇒ lời giải duy nhất cũng là lời giải ngắn nhất.
3. Điều kiện khả thi: tổng số nút lệch phải **chẵn** (mỗi thao tác đổi đúng hai nút).
4. Cài đặt bằng DFS hậu thứ tự **lặp**, bằng "bóc lá" từ ngoài vào, hoặc bằng thứ tự BFS ngược.
5. Kiểm thử với vét cạn mọi tập cạnh.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cây `n` nút, cạnh `edges[i] = [a, b]`. Mỗi nút có màu ban đầu `start[x]` và màu mong muốn `target[x]` (`'0'`/`'1'`). Một thao tác: chọn cạnh `i` và **lật màu cả hai đầu**. Trả về danh sách chỉ số cạnh (tăng dần) của một dãy thao tác **ngắn nhất** biến `start` thành `target`; nếu không thể, trả về `[-1]`.

**Ví dụ 1:** `n = 3`, cạnh `[[0,1],[1,2]]`, `start = "010"`, `target = "100"` → `[0]`.

**Ví dụ 2:** `n = 7`, cạnh `[[0,1],[1,2],[2,3],[3,4],[3,5],[1,6]]`, `start = "0011000"`, `target = "0010001"` → `[1,2,5]`.

**Ví dụ 3:** `n = 2`, `start = "00"`, `target = "01"` → `[-1]`.

**Ràng buộc:** `2 ≤ n ≤ 10^5`.

In [1]:
from typing import List
from collections import deque
from itertools import combinations
import random, time

examples = [
    (3, [[0, 1], [1, 2]], "010", "100", [0]),
    (7, [[0, 1], [1, 2], [2, 3], [3, 4], [3, 5], [1, 6]], "0011000", "0010001", [1, 2, 5]),
    (2, [[0, 1]], "00", "01", [-1]),
]
for n, e, s, t, exp in examples:
    print(n, e, s, t, "->", exp)

3 [[0, 1], [1, 2]] 010 100 -> [0]
7 [[0, 1], [1, 2], [2, 3], [3, 4], [3, 5], [1, 6]] 0011000 0010001 -> [1, 2, 5]
2 [[0, 1]] 00 01 -> [-1]


## 3. Kiến thức nền

### 3.1. Lật màu = XOR với 1; thứ tự không quan trọng

Gọi `d[x] = start[x] ⊕ target[x]` ("nút lệch"). Một thao tác trên cạnh `(u, v)` XOR `1` vào `d[u]`, `d[v]`. Phép XOR giao hoán ⇒ thứ tự thao tác không quan trọng; dùng một cạnh hai lần ⇒ không đổi gì ⇒ chỉ cần chọn một **tập** cạnh.

In [2]:
d = [int(a) ^ int(b) for a, b in zip("0011000", "0010001")]
print("nút lệch d =", d, "| tổng =", sum(d))

nút lệch d = [0, 0, 0, 1, 0, 0, 1] | tổng = 2


### 3.2. Điều kiện chẵn lẻ

Mỗi thao tác đổi trạng thái lệch của đúng **hai** nút ⇒ tính chẵn lẻ của `Σ d[x]` không đổi. Muốn về `0` ⇒ `Σ d` phải chẵn. Ví dụ 3: `d = [0, 1]` lẻ ⇒ `[-1]`.

### 3.3. DFS hậu thứ tự lặp (không đệ quy)

Dùng ngăn xếp với cờ "đã duyệt con chưa": lần đầu gặp nút thì đẩy lại chính nó (cờ = 1) rồi đẩy các con; lần thứ hai (cờ = 1) mới xử lý nút — lúc đó mọi con đã xong.

In [3]:
def postorder(n, adj, root=0):
    order, st = [], [(root, -1, 0)]
    while st:
        u, p, done = st.pop()
        if done:
            order.append(u); continue
        st.append((u, p, 1))
        for v, _ in adj[u]:
            if v != p:
                st.append((v, u, 0))
    return order

adj = [[] for _ in range(4)]
for i, (a, b) in enumerate([[0, 1], [1, 2], [1, 3]]):
    adj[a].append((b, i)); adj[b].append((a, i))
print(postorder(4, adj))

[3, 2, 1, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Nhìn một **lá** `x` (nối với cha `p` bằng cạnh `e`). Chỉ cạnh `e` chạm tới `x`. Vậy:

- nếu `x` lệch ⇒ **bắt buộc** lật `e` (và `p` bị lật theo);
- nếu `x` không lệch ⇒ **bắt buộc không** lật `e`.

Xử lý xong lá, "cắt" lá đi; cây còn lại có lá mới, lặp lại. Mỗi quyết định là **bắt buộc** ⇒ lời giải (nếu có) **duy nhất** ⇒ tự động ngắn nhất.

Tính tay ví dụ 1 (đường `0 - 1 - 2`, `d = [1, 1, 0]`): lá `2` không lệch ⇒ không lật cạnh 1. Lá tiếp theo `1` (sau khi cắt 2) lệch ⇒ lật cạnh 0 ⇒ `d[0]` thành 0. Gốc `0` không lệch ⇒ khả thi. Đáp án `[0]` ✔.

## 5. Cách 1 — Brute Force: thử mọi tập cạnh

Duyệt tập cạnh theo kích thước tăng dần; tập đầu tiên làm mọi `d[x] = 0` là đáp án (trong cùng kích thước, duyệt theo thứ tự `combinations` cho tập nhỏ nhất theo từ điển — nhưng trên cây chỉ có **một** tập hợp lệ).

In [4]:
def brute(n, edges, start, target):
    d0 = [int(a) ^ int(b) for a, b in zip(start, target)]
    m = len(edges)
    for size in range(m + 1):
        for S in combinations(range(m), size):
            d = d0[:]
            for i in S:
                a, b = edges[i]; d[a] ^= 1; d[b] ^= 1
            if not any(d):
                return list(S)
    return [-1]

for n, e, s, t, exp in examples:
    print(brute(n, e, s, t), "kỳ vọng", exp)

[0] kỳ vọng [0]
[1, 2, 5] kỳ vọng [1, 2, 5]
[-1] kỳ vọng [-1]


**Nút thắt:** `2^{n−1}` tập. Mục 4 cho thấy mọi quyết định đều bị ép buộc từ lá lên ⇒ `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Định lý (cây).** Gốc tại `0`. Cạnh nối `v` với cha được lật **khi và chỉ khi** số nút lệch trong cây con của `v` là **lẻ**.

*Lý do:* mọi cạnh **bên trong** cây con của `v` đổi trạng thái lệch của hai nút **trong** cây con ⇒ không đổi tính chẵn lẻ của tổng lệch trong cây con. Chỉ cạnh `(v, cha)` đổi đúng một nút trong cây con (`v`). Muốn tổng lệch trong cây con về 0 (chẵn) ⇒ cạnh này phải lật đúng khi tổng ban đầu lẻ.

**Thuật toán:** tính `need[v]` = tính chẵn lẻ số nút lệch trong cây con của `v` (hậu thứ tự: `need[v] = d[v] ⊕ (XOR need[con])`). Nếu `need[v] = 1` (và `v ≠ gốc`) ⇒ chọn cạnh `(v, cha)`. Nếu `need[gốc] = 1` ⇒ tổng lệch lẻ ⇒ `[-1]`.

Do lời giải duy nhất, chỉ cần **sắp xếp** chỉ số cạnh để trả về theo thứ tự tăng.

## 7. Các cách tiếp theo

- **Cách 2:** **bóc lá** (BFS từ các lá vào trong như sắp xếp tô-pô) — mô phỏng trực giác mục 4.
- **Cách 3:** DFS hậu thứ tự lặp tính `need[v]` (lời giải nộp chính).
- **Cách 4:** BFS từ gốc lấy thứ tự + cha, rồi xử lý theo **thứ tự BFS ngược** (con luôn trước cha).

### 7.1. Cách 2 — Bóc lá

In [5]:
def peel_leaves(n, edges, start, target):
    d = [int(a) ^ int(b) for a, b in zip(start, target)]
    if sum(d) % 2:
        return [-1]
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(edges):
        adj[a].append((b, i)); adj[b].append((a, i))
    deg = [len(adj[x]) for x in range(n)]
    removed = [False] * n
    q = deque(x for x in range(n) if deg[x] == 1)
    chosen = []
    while q:
        x = q.popleft()
        if removed[x] or deg[x] != 1:
            continue
        removed[x] = True
        for y, i in adj[x]:
            if not removed[y]:                   # láng giềng duy nhất còn lại
                if d[x]:
                    chosen.append(i); d[x] ^= 1; d[y] ^= 1
                deg[y] -= 1
                if deg[y] == 1:
                    q.append(y)
    return sorted(chosen)

for n, e, s, t, exp in examples:
    print(peel_leaves(n, e, s, t), "kỳ vọng", exp)

[0] kỳ vọng [0]
[1, 2, 5] kỳ vọng [1, 2, 5]
[-1] kỳ vọng [-1]


**Độ phức tạp:** mỗi nút bị bóc một lần, mỗi cạnh xét hai lần ⇒ `O(n)` (+ `O(n log n)` sắp xếp). Lưu ý nút cuối cùng còn lại có bậc 0 và không bị đưa vào hàng đợi — nó chính là "gốc", và nhờ điều kiện tổng chẵn, nó tự hết lệch.

### 7.2. Cách 3 — DFS hậu thứ tự lặp (lời giải chính)

**Giải thích từng đoạn:**

1. Danh sách kề lưu `(láng giềng, chỉ số cạnh)`.
2. Ngăn xếp `(u, cha, cạnh_tới_cha, done)`. Khi `done`: `need[u] = d[u] ⊕ XOR need[con]`; nếu `need[u]` và `u` không phải gốc ⇒ chọn `cạnh_tới_cha`.
3. Nếu `need[0]` ⇒ `[-1]`; ngược lại trả về danh sách đã sắp.

In [6]:
def dfs_postorder(n, edges, start, target):
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(edges):
        adj[a].append((b, i)); adj[b].append((a, i))
    need = [0] * n
    chosen = []
    st = [(0, -1, -1, False)]
    while st:
        u, p, pe, done = st.pop()
        if not done:
            st.append((u, p, pe, True))
            for v, i in adj[u]:
                if v != p:
                    st.append((v, u, i, False))
            continue
        x = int(start[u] != target[u])
        for v, i in adj[u]:
            if v != p:
                x ^= need[v]
        need[u] = x
        if x and p != -1:
            chosen.append(pe)
    if need[0]:
        return [-1]
    return sorted(chosen)

for n, e, s, t, exp in examples:
    print(dfs_postorder(n, e, s, t), "kỳ vọng", exp)

[0] kỳ vọng [0]
[1, 2, 5] kỳ vọng [1, 2, 5]
[-1] kỳ vọng [-1]


**Độ phức tạp Cách 3:** `O(n)` duyệt + `O(n log n)` sắp xếp (hoặc `O(n)` nếu đánh dấu mảng boolean theo chỉ số cạnh rồi lọc).

### 7.3. Cách 4 — Thứ tự BFS ngược

In [7]:
def bfs_reverse(n, edges, start, target):
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(edges):
        adj[a].append((b, i)); adj[b].append((a, i))
    parent, pedge = [-1] * n, [-1] * n
    order = [0]; seen = [False] * n; seen[0] = True
    for u in order:                                  # BFS (list mở rộng dần)
        for v, i in adj[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; pedge[v] = i; order.append(v)
    need = [int(a != b) for a, b in zip(start, target)]
    mark = [False] * (n - 1)
    for u in reversed(order):
        if u != 0 and need[u]:
            mark[pedge[u]] = True
            need[parent[u]] ^= 1                     # cạnh lật làm cha đổi trạng thái
    if need[0]:
        return [-1]
    return [i for i in range(n - 1) if mark[i]]      # đã tăng dần, không cần sort

for n, e, s, t, exp in examples:
    print(bfs_reverse(n, e, s, t), "kỳ vọng", exp)

[0] kỳ vọng [0]
[1, 2, 5] kỳ vọng [1, 2, 5]
[-1] kỳ vọng [-1]


In [8]:
class SolutionBFS:
    def minimumFlips(self, n, edges, start, target):
        return bfs_reverse(n, edges, start, target)

print([SolutionBFS().minimumFlips(*ex[:4]) for ex in examples])

[[0], [1, 2, 5], [-1]]


**Độ phức tạp Cách 4:** `O(n)` hoàn toàn (đánh dấu theo chỉ số cạnh thay cho sắp xếp).

## 8. TRACE từng bước

Trace Cách 4 trên ví dụ 2. Cây (gốc 0): `0 - 1`, `1 - 2`, `2 - 3`, `3 - 4`, `3 - 5`, `1 - 6`. `d = [0,0,0,1,0,0,1]`.

In [9]:
def trace(n, edges, start, target):
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(edges):
        adj[a].append((b, i)); adj[b].append((a, i))
    parent, pedge = [-1] * n, [-1] * n
    order = [0]; seen = [False] * n; seen[0] = True
    for u in order:
        for v, i in adj[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; pedge[v] = i; order.append(v)
    need = [int(a != b) for a, b in zip(start, target)]
    print("thứ tự BFS:", order, "| d =", need)
    chosen = []
    for u in reversed(order):
        if u == 0: continue
        if need[u]:
            chosen.append(pedge[u]); need[parent[u]] ^= 1
            print(f"nút {u}: lệch -> lật cạnh {pedge[u]} ({u}-{parent[u]}), d[{parent[u]}] = {need[parent[u]]}")
        else:
            print(f"nút {u}: không lệch -> giữ cạnh {pedge[u]}")
    print("gốc còn lệch?", need[0], "-> đáp án", sorted(chosen) if not need[0] else [-1])

trace(*examples[1][:4])

thứ tự BFS: [0, 1, 2, 6, 3, 4, 5] | d = [0, 0, 0, 1, 0, 0, 1]
nút 5: không lệch -> giữ cạnh 4
nút 4: không lệch -> giữ cạnh 3
nút 3: lệch -> lật cạnh 2 (3-2), d[2] = 1
nút 6: lệch -> lật cạnh 5 (6-1), d[1] = 1
nút 2: lệch -> lật cạnh 1 (2-1), d[1] = 0
nút 1: không lệch -> giữ cạnh 0
gốc còn lệch? 0 -> đáp án [1, 2, 5]


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minimumFlips(self, n: int, edges: List[List[int]], start: str, target: str) -> List[int]:
        adj = [[] for _ in range(n)]
        for i, (a, b) in enumerate(edges):
            adj[a].append((b, i))
            adj[b].append((a, i))
        parent = [-1] * n
        pedge = [-1] * n
        order = [0]
        seen = [False] * n
        seen[0] = True
        for u in order:
            for v, i in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    parent[v] = u
                    pedge[v] = i
                    order.append(v)
        need = [start[x] != target[x] for x in range(n)]
        mark = [False] * (n - 1)
        for u in reversed(order):
            if u and need[u]:
                mark[pedge[u]] = True
                need[parent[u]] = not need[parent[u]]
        if need[0]:
            return [-1]
        return [i for i in range(n - 1) if mark[i]]

sol = Solution()
for n, e, s, t, exp in examples:
    got = sol.minimumFlips(n, e, s, t)
    print(got, "✔" if got == exp else "✘")

[0] ✔
[1, 2, 5] ✔
[-1] ✔


## 10. Vì sao thuật toán đúng

1. **Tập cạnh:** vì phép lật là XOR, mọi dãy thao tác tương đương với tập cạnh dùng **lẻ** lần; độ dài dãy ≥ kích thước tập. Nên chỉ cần tìm tập cạnh nhỏ nhất.
2. **Duy nhất:** với gốc cố định, xét cạnh `(v, cha(v))`. Các cạnh khác chạm cây con `T(v)` hoặc nằm hẳn trong `T(v)` (đổi hai nút trong `T(v)`) hoặc hẳn ngoài `T(v)` (không đổi nút nào trong `T(v)`). Vậy tính chẵn lẻ của số nút lệch còn lại trong `T(v)` chỉ phụ thuộc việc lật cạnh `(v, cha)` ⇒ cạnh này **phải** lật ⇔ số nút lệch ban đầu trong `T(v)` lẻ. Mọi cạnh bị xác định ⇒ tối đa một tập hợp lệ.
3. **Khả thi:** tập vừa xác định làm mọi nút khác gốc hết lệch (quy nạp từ lá); gốc hết lệch ⇔ tổng lệch chẵn. Ngược lại không có tập nào ⇒ `[-1]`.
4. **Cài đặt:** xử lý theo thứ tự BFS ngược bảo đảm mọi con được xử lý trước cha; `need[u]` khi xét chính là tính chẵn lẻ số nút lệch trong `T(u)`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn tập cạnh | `O(2^{n−1} · n)` | `O(n)` |
| 2. Bóc lá | `O(n)` + sắp xếp | `O(n)` |
| 3. DFS hậu thứ tự | `O(n)` + sắp xếp | `O(n)` |
| 4. BFS ngược + đánh dấu | `O(n)` | `O(n)` |

## 12. Bộ test

In [11]:
def run_all(n, e, s, t, with_brute=True):
    res = [peel_leaves(n, e, s, t), dfs_postorder(n, e, s, t), bfs_reverse(n, e, s, t), Solution().minimumFlips(n, e, s, t)]
    if with_brute:
        res.append(brute(n, e, s, t))
    return res

for n, e, s, t, exp in examples:
    assert all(r == exp for r in run_all(n, e, s, t))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    (2, [[0, 1]], "00", "00", []),           # không cần thao tác
    (2, [[0, 1]], "01", "10", [0]),
    (4, [[0, 1], [0, 2], [0, 3]], "1111", "0000", [0, 1, 2]),   # sao: cả ba lá lật cạnh của mình, tâm bị lật 3 lần -> hết lệch
    (4, [[0, 1], [1, 2], [2, 3]], "1001", "0000", [0, 1, 2]),
]
for n, e, s, t, exp in cases:
    r = run_all(n, e, s, t)
    print((n, e, s, t), "->", r[0], "| vét cạn:", r[-1])
    assert all(x == r[-1] == exp for x in r)

(2, [[0, 1]], '00', '00') -> [] | vét cạn: []
(2, [[0, 1]], '01', '10') -> [0] | vét cạn: [0]
(4, [[0, 1], [0, 2], [0, 3]], '1111', '0000') -> [0, 1, 2] | vét cạn: [0, 1, 2]
(4, [[0, 1], [1, 2], [2, 3]], '1001', '0000') -> [0, 1, 2] | vét cạn: [0, 1, 2]


In [13]:
# 12.3 phản ví dụ: chỉ kiểm tra tổng lệch chẵn rồi ghép cặp nút lệch "kề nhau" theo chỉ số là sai
def wrong_pair_by_index(n, edges, start, target):
    d = [i for i in range(n) if start[i] != target[i]]
    if len(d) % 2: return [-1]
    idx = {tuple(sorted(e)): i for i, e in enumerate(edges)}
    out = []
    for a, b in zip(d[::2], d[1::2]):
        if (min(a, b), max(a, b)) in idx: out.append(idx[(min(a, b), max(a, b))])
        else: return "không ghép được trực tiếp"
    return sorted(out)

print("sai:", wrong_pair_by_index(*examples[1][:4]), "| đúng:", Solution().minimumFlips(*examples[1][:4]))

sai: không ghép được trực tiếp | đúng: [1, 2, 5]


In [14]:
# 12.4 đối chiếu ngẫu nhiên
def random_tree(rng, n):
    perm = list(range(n)); rng.shuffle(perm)
    return [[perm[rng.randrange(i)], perm[i]] for i in range(1, n)]

rng = random.Random(3812)
for _ in range(300):
    n = rng.randint(2, 10); e = random_tree(rng, n)
    s = "".join(rng.choice("01") for _ in range(n)); t = "".join(rng.choice("01") for _ in range(n))
    r = run_all(n, e, s, t)
    assert all(x == r[-1] for x in r), (n, e, s, t, r)
print("300 test ngẫu nhiên (có vét cạn): OK")

300 test ngẫu nhiên (có vét cạn): OK


In [15]:
# 12.5 test lớn: cây đường thẳng (sâu) và cây ngẫu nhiên n = 10^5; kiểm tra áp dụng thao tác cho ra target
def apply_ops(n, edges, start, ops):
    c = [int(x) for x in start]
    for i in ops:
        a, b = edges[i]; c[a] ^= 1; c[b] ^= 1
    return "".join(map(str, c))

rng = random.Random(9)
n = 10**5
for name, e in (("đường thẳng", [[i, i + 1] for i in range(n - 1)]), ("ngẫu nhiên", random_tree(rng, n))):
    s = "".join(rng.choice("01") for _ in range(n)); t = list("".join(rng.choice("01") for _ in range(n)))
    if sum(a != b for a, b in zip(s, t)) % 2: t[0] = "1" if t[0] == "0" else "0"
    t = "".join(t)
    t0 = time.time(); ops = Solution().minimumFlips(n, e, s, t); t1 = time.time()
    print(f"{name}: {len(ops)} cạnh, đúng target: {apply_ops(n, e, s, ops) == t}, khớp DFS: {ops == dfs_postorder(n, e, s, t)}, {t1 - t0:.2f}s")

đường thẳng: 50118 cạnh, đúng target: True, khớp DFS: True, 0.11s


ngẫu nhiên: 49949 cạnh, đúng target: True, khớp DFS: True, 0.21s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — DFS đệ quy trên cây sâu

Cây đường thẳng `10^5` nút ⇒ đệ quy sâu `10^5` ⇒ `RecursionError` (hoặc tràn ngăn xếp C khi tăng giới hạn). **Sửa:** DFS lặp hoặc BFS ngược như Cách 3/4.

In [16]:
import sys
def dfs_rec(u, p, adj, need, start, target, chosen, pe):
    x = int(start[u] != target[u])
    for v, i in adj[u]:
        if v != p:
            x ^= dfs_rec(v, u, adj, need, start, target, chosen, i)
    if x and p != -1: chosen.append(pe)
    return x

m = 5000
adj = [[] for _ in range(m)]
for i in range(m - 1): adj[i].append((i + 1, i)); adj[i + 1].append((i, i))
try:
    dfs_rec(0, -1, adj, None, "1" * m, "0" * m, [], -1); print("không lỗi")
except RecursionError:
    print("RecursionError với cây sâu", m, "(giới hạn", sys.getrecursionlimit(), ")")

RecursionError với cây sâu 5000 (giới hạn 1000 )


### Lỗi 2 — Quên cập nhật trạng thái của **cha** khi lật cạnh

In [17]:
def wrong_no_parent_update(n, edges, start, target):
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(edges):
        adj[a].append((b, i)); adj[b].append((a, i))
    parent, pedge = [-1] * n, [-1] * n; order = [0]; seen = [False] * n; seen[0] = True
    for u in order:
        for v, i in adj[u]:
            if not seen[v]: seen[v] = True; parent[v] = u; pedge[v] = i; order.append(v)
    need = [start[x] != target[x] for x in range(n)]; out = []
    for u in reversed(order):
        if u and need[u]: out.append(pedge[u])          # SAI: không lật need[parent[u]]
    return [-1] if need[0] else sorted(out)

print("sai:", wrong_no_parent_update(*examples[0][:4]), "| đúng:", Solution().minimumFlips(*examples[0][:4]))

sai: [-1] | đúng: [0]


### Lỗi 3 — Không kiểm tra khả thi (gốc còn lệch)

Nếu bỏ qua `need[0]`, ví dụ 3 trả về `[0]` — áp dụng thì được `"11"` chứ không phải `"01"`.

In [18]:
def wrong_no_check(n, edges, start, target):
    r = Solution().minimumFlips(n, edges, start, target)
    if r == [-1]:
        # mô phỏng code quên kiểm tra: vẫn trả về các cạnh đã chọn
        return [0] if n == 2 else r
    return r

ops = wrong_no_check(*examples[2][:4])
print("trả về", ops, "-> áp dụng ra", apply_ops(2, [[0, 1]], "00", ops), "≠ target '01'")

trả về [0] -> áp dụng ra 11 ≠ target '01'


### Lỗi 4 — Trả về chỉ số cạnh theo thứ tự chọn (không sắp xếp)

Đề yêu cầu **tăng dần**. Thứ tự hậu thứ tự/BFS ngược thường không tăng.

In [19]:
def unsorted_version(n, edges, start, target):
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(edges):
        adj[a].append((b, i)); adj[b].append((a, i))
    parent, pedge = [-1] * n, [-1] * n; order = [0]; seen = [False] * n; seen[0] = True
    for u in order:
        for v, i in adj[u]:
            if not seen[v]: seen[v] = True; parent[v] = u; pedge[v] = i; order.append(v)
    need = [start[x] != target[x] for x in range(n)]; out = []
    for u in reversed(order):
        if u and need[u]: out.append(pedge[u]); need[parent[u]] = not need[parent[u]]
    return out                                            # SAI: chưa sắp xếp

print("chưa sắp:", unsorted_version(*examples[1][:4]), "| đúng:", Solution().minimumFlips(*examples[1][:4]))

chưa sắp: [2, 5, 1] | đúng: [1, 2, 5]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Thao tác lật/XOR** ⇒ mỗi thao tác dùng 0/1 lần, thứ tự không quan trọng.
- **Trên cây, ràng buộc tại lá bị ép buộc** ⇒ tham lam từ lá lên (hậu thứ tự / bóc lá), thường cho lời giải **duy nhất**.
- **Bất biến chẵn lẻ** (mỗi thao tác đổi đúng hai nút) ⇒ điều kiện khả thi nhanh.
- Liên quan: LC 2440 *Create Components With Same Value*, LC 979 *Distribute Coins in Binary Tree* (đẩy "nợ" lên cha), bài "Lights Out" trên cây.

## 15. Biến thể của bài

1. **Đồ thị tổng quát** (có chu trình): không còn duy nhất; tập cạnh hợp lệ tồn tại ⇔ mỗi thành phần liên thông có số nút lệch chẵn; tối thiểu số cạnh = ghép cặp nút lệch bằng đường đi (T-join) — khó hơn nhiều.
2. **Lật một nút và mọi láng giềng của nó** ("Lights Out"): hệ phương trình tuyến tính trên GF(2); trên cây có DP.
3. **Mỗi cạnh có chi phí**: vì lời giải duy nhất, chi phí cũng xác định.
4. **Đếm số nút cần lật trên đường đi** giữa hai nút cho trước (truy vấn): kết hợp với LCA.

In [20]:
# Biến thể 1 (kiểm tra khả thi trên đồ thị tổng quát): mỗi thành phần liên thông phải có số nút lệch chẵn
def feasible_general(n, edges, start, target):
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for a, b in edges: parent[find(a)] = find(b)
    from collections import Counter
    c = Counter(find(x) for x in range(n) if start[x] != target[x])
    return all(v % 2 == 0 for v in c.values())

print(feasible_general(4, [[0, 1], [1, 2], [2, 0]], "1100", "0000"), feasible_general(4, [[0, 1], [2, 3]], "1010", "0000"))

True False


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi tập cạnh | `O(2^n n)` | `O(n)` | Kiểm thử |
| 2. Bóc lá | Lá bị ép quyết định | `O(n log n)` | `O(n)` | Trực quan |
| 3. DFS hậu thứ tự | Chẵn lẻ cây con | `O(n log n)` | `O(n)` | Khuôn DFS cây |
| 4. BFS ngược + đánh dấu | Con trước cha, không cần sort | `O(n)` | `O(n)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `n = 3`, cạnh `[[0,1],[0,2]]`, `start = "011"`, `target = "000"` → ?

**Bài 2 (tính tay).** Đường `0-1-2-3-4`, `start = "10001"`, `target = "00000"`.

**Bài 3 (điền TODO).** Viết `count_flips(n, edges, start, target)` trả về **số** cạnh cần lật (hoặc `-1`).

**Bài 4 (tự viết lại).** Viết Cách 3 dùng `need` nhưng gốc tại nút `n − 1` thay vì 0 — kết quả không đổi (vì lời giải duy nhất).

**Bài 5 (mini challenge).** Với `q` truy vấn, mỗi truy vấn đổi `target[x]` và hỏi số cạnh cần lật. Gợi ý: cạnh `(v, cha)` lật ⇔ chẵn lẻ cây con lẻ ⇒ đổi `target[x]` lật chẵn lẻ của mọi tổ tiên của `x` ⇒ cần cấu trúc trên đường đi (hoặc làm `O(n)` mỗi truy vấn cho bài nhỏ).

In [21]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_flips(n, edges, start, target):
    # TODO: dùng Solution, trả về len(...) hoặc -1
    pass

In [22]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def flips_after_updates(n, edges, start, target, updates):
    # TODO: với mỗi cập nhật x: đảo target[x], tính lại số cạnh cần lật
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `d = [0, 1, 1]`, tổng chẵn. (2) Lá 1 lệch ⇒ lật cạnh 0 ⇒ gốc lệch; lá 2 lệch ⇒ lật cạnh 1 ⇒ gốc hết lệch. (3) `[0, 1]`.

**Bài 2.** (1) `d = [1,0,0,0,1]`. (2) Từ lá 4: lật cạnh 3, d[3] = 1; nút 3 lật cạnh 2; nút 2 lật cạnh 1; nút 1 lật cạnh 0 ⇒ d[0] = 0. (3) `[0, 1, 2, 3]`.

**Bài 3.** (1) Gọi `Solution`. (2) `[-1]` ⇒ `-1`. (3) Ngược lại `len`.

**Bài 4.** (1) Đổi gốc của BFS. (2) Kiểm tra `need[gốc]`. (3) So kết quả với gốc 0.

**Bài 5.** (1) Bản đơn giản: chạy lại `O(n)` mỗi truy vấn. (2) Bản nhanh: số cạnh = số nút `v ≠ gốc` có chẵn lẻ cây con lẻ; đổi `target[x]` lật chẵn lẻ các nút trên đường `x → gốc`. (3) HLD + segment tree đếm số 1 có lật đoạn.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().minimumFlips(3, [[0, 1], [0, 2]], "011", "000") == [0, 1]
assert Solution().minimumFlips(5, [[0, 1], [1, 2], [2, 3], [3, 4]], "10001", "00000") == [0, 1, 2, 3]
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [24]:
# Bài 3, 4
def count_flips_sol(n, edges, start, target):
    r = Solution().minimumFlips(n, edges, start, target)
    return -1 if r == [-1] else len(r)

def rooted_at(n, edges, start, target, root):
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(edges):
        adj[a].append((b, i)); adj[b].append((a, i))
    parent, pedge = [-1] * n, [-1] * n; order = [root]; seen = [False] * n; seen[root] = True
    for u in order:
        for v, i in adj[u]:
            if not seen[v]: seen[v] = True; parent[v] = u; pedge[v] = i; order.append(v)
    need = [start[x] != target[x] for x in range(n)]; mark = [False] * (n - 1)
    for u in reversed(order):
        if u != root and need[u]: mark[pedge[u]] = True; need[parent[u]] = not need[parent[u]]
    return [-1] if need[root] else [i for i in range(n - 1) if mark[i]]

rng = random.Random(4)
for _ in range(300):
    n = rng.randint(2, 12); e = random_tree(rng, n)
    s = "".join(rng.choice("01") for _ in range(n)); t = "".join(rng.choice("01") for _ in range(n))
    assert rooted_at(n, e, s, t, n - 1) == Solution().minimumFlips(n, e, s, t)
print("Bài 3:", count_flips_sol(*examples[1][:4]), "| Bài 4 OK (đổi gốc không đổi kết quả)")

Bài 3: 3 | Bài 4 OK (đổi gốc không đổi kết quả)


In [25]:
# Bài 5 (bản O(n) mỗi truy vấn)
def flips_after_updates_sol(n, edges, start, target, updates):
    t = list(target); out = []
    for x in updates:
        t[x] = "1" if t[x] == "0" else "0"
        out.append(count_flips_sol(n, edges, start, "".join(t)))
    return out

print("Bài 5:", flips_after_updates_sol(*examples[1][:4], [0, 0, 6, 3]))

Bài 5: [-1, 3, -1, 0]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
BFS từ 0: order, parent, pedge
need[x] = start[x] != target[x]
for u in reversed(order):            # con trước cha
    if u != 0 and need[u]:
        mark[pedge[u]] = True
        need[parent[u]] ^= True
return [-1] if need[0] else [i for i in range(n-1) if mark[i]]
```

**Checklist:**

- [ ] Lật cạnh thì **cha** đổi trạng thái lệch?
- [ ] Kiểm tra gốc còn lệch ⇒ `[-1]`?
- [ ] Chỉ số cạnh trả về **tăng dần**?
- [ ] Không đệ quy sâu (cây đường thẳng `10^5`)?
- [ ] Không cần "tối thiểu hóa" riêng — lời giải trên cây là duy nhất?